In [ ]:
import pandas as pd

Import Pandas library for continued use throughout model.

In [ ]:
df = pd.read_csv("../data/creditsense_applications.csv")

Import original dataset for continued use throughout model.

In [ ]:
df.shape

Confirm dataset is returning the correct amount of rows and columns.

In [ ]:
df[["savings_balance_gbp", "years_at_current_address"]].isna().sum()

Confirming totals for missing records before filtering data for model build. With the numbers recorded above and information from investigations we know a total of 418, after accounting for entries missing both, will need to be handled. Please refer to `decisions.md` decision 002 for further logic.

In [ ]:
df_model = df.dropna(
    subset=["savings_balance_gbp", "years_at_current_address"]
).copy()

Creating a seperate dataframe for modelling in order to preserve our original dataframe. Specifically targetting the two columns we know are missing values means we protect against changing behaviour should the dataset change in future.

In [ ]:
df_model.shape

Confirming the removal of missing values from above has been successful and we have not lost any extras in the process.

In [ ]:
df_model.columns.tolist()

Checking the columns to confirm targets, identifiers and predictors.

In [ ]:
print("Dataset shape:", df_model.shape)
print("\nMissing values:", df_model.isna().sum().sum())
print("\nDefault distribution:")
print(df_model["default_within_24m"].value_counts(normalize=True))

Confirming all data has been successfully retained after a few days away from the project.

In [ ]:
excluded_features = [
    "application_id",
    "default_within_24m",
    "age_years",
    "sex",
    "marital_status",
    "residency_status",
    "num_dependents",
    "region",
    "loan_purpose",
    "application_channel",
    "previous_application_declined"
]

Creating a clear but changeable list of those features I cannot include like targets and identifiers along with ones which will be excluded for this experiment but may be changed in further iterations of the model.

In [ ]:
X = df_model.drop(columns=excluded_features)

y = df_model["default_within_24m"]

print("predictive features:", X.shape[1])
print("Target:", y.name)
print("\nSelected features:")
print(X.columns.tolist())

Created our first groups and ran print commands to ensure those have been successful before deciding handling for each feature.

In [ ]:
X.dtypes

Confirming data types before applying handling decisions.

In [ ]:
categorical_cols = X.select_dtypes(include=["str", "object"]).columns

for col in categorical_cols:
    print(f"\n{col}")
    print(X[col].value_counts())

Getting a breakdown of the features to make final handling decisions in a clear way.

In [ ]:
nominal_features = [
    "employment_status",
    "housing_status",
    "credit_history",
    "checking_account_status"
]

binary_features = [
    "has_savings_account",
    "has_guarantor"
]

numerical_features = [
    col for col in X.columns
    if col not in nominal_features + binary_features
]

Splitting the features into their eventual groups for handling.

In [ ]:
print("Nominal:", nominal_features)
print("Binary:", binary_features)
print("Numerical:", numerical_features)

print("\nTotal features:",
      len(nominal_features)
      + len(binary_features)
      + len(numerical_features))

Confirming the split has been successful.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

Creating the train and test groups ensuring the default rates are maintained at the same proportion in both sets. Chose an 80/20 split to give us enough training data given the amount of things we are investigating.

In [ ]:
print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

print("\nTraining default rate:")
print(y_train.value_counts(normalize=True))

print("\nTesting default rate:")
print(y_test.value_counts(normalize=True))

Verifying the split. Note: Test data will be held back for independant evaluation. Preprocessing will only be fitted to training data to avoid leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    OrdinalEncoder
)

Importing the required transformers.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("nom", OneHotEncoder(handle_unknown="ignore"), nominal_features),
        ("bin", OrdinalEncoder(
            categories=[["No", "Yes"], ["No", "Yes"]]
        ), binary_features)
    ]
)

Set up for the preprocessing. This has not seen any data yet, it is simply to define each of the transformations in advance.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

Import the pipeline and model type.

In [ ]:
baseline_model = Pipeline(
    steps =[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

Created baseline model, but have not fitted it to any data yet.

In [ ]:
baseline_model.fit(X_train, y_train)

First round of model training. Pipeline is exactly as expected and has given me plenty to look further into for further iterations already.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc"
)

print("ROC_AUC per fold:", cv_scores)
print("Mean ROAC_AUC:", cv_scores.mean())
print("Standard deviation:", cv_scores.std())

First run through showed a mean ROC-AUC of 0.8212 with a small variation accross folds. Further evaluation will be required, but this is a good start.

In [ ]:
y_pred = baseline_model.predict(X_test)

print("Predictions:", y_pred[:10])
print("Total predictions:", len(y_pred))

Successful generation of prediction for the held-out test set. Confirmed correct number of testing records against annotations. This is with a default 0.5 threshold.

In [ ]:
import numpy as np
unique, counts = np.unique(y_pred, return_counts=True)
print(dict(zip(unique, counts)))

Confirmed the predicted default rate in the test set, returning 933 won't default and 184 will default. This amounts to a 16.% default rate according to the model used here.

In [ ]:
print(y_test.value_counts())

Confirmed the amount of applicants who actually defaulted. This difference does not prove underfitting as we have not broken down the results yet.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

Confusion matrix used to determine effectiveness against stakeholder needs. The model correctly identifies 92.5% of non-defaults correctly, but only 42.5% of defaults correctly. An incorrect prediction of no-default could be quite damaging to the stakeholder. (False Negatives being a priority!)

### Stats Defined so far:
- Accuracy: 79.9% correct predictions.
- Default recall: 42.5%

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

Ascertaining F1 score for this model shows a relatively good score for predicting who will not default and then a considerably worse score for predicting those who will default. The default recall as described above is bringing the F1 score for those who will default down. Looking at the F1 score alone though can obscure class-specific performance differences. We must always look at the data as a whole before making recommendations. 

In [ ]:
from sklearn.metrics import roc_auc_score
y_proba = baseline_model.predict_proba(X_test)[:, 1]
test_roc_auc = roc_auc_score(y_test, y_proba)
print("Test ROC_AUC:", round(test_roc_auc, 4))

Using `[:, 1]` to estimate probability of class 1 (default) for each applicant. With a result of 0.8129 we can say that the model has a fairly strong ability to rank applicants by their estimated default risk. However with the threshold at our baseline of 0.5 we have a default recall of 42.5% meaning the model misses more than half of the applicant pool that actually end up defaulting.